# Dashed-line detection lab: collinear grouping

Extract one page's raw vectors and split them into **layer -> color -> stroke-width** buckets. This is the P3
VectorClassification separation, but *without* the seqno merge or spatial clustering.

Within each bucket, take the **straight** Vectors: all items are `'l'`, and every point is within
`STRAIGHT_TOL_PT` of one line. Group the ones that lie on the **same infinite line**:
- angle within `ANGLE_TOL_DEG`;
- perpendicular offset within `OFFSET_TOL_PT`;
- no limit on the gap along the line.

A dashed line should show up as one group of many short vectors of similar length.

Layers (open with `scripts/pipeline_report_viewer.py`):
- `collinear / groups (distinct colour)`: each group with >= 2 members in its own hue.
- `gradient / vector count`: fewest -> most members, coloured green -> yellow -> red -> purple at
  `(n - min) / (max - min)`.
- `gradient / length std`: most -> least uniform lengths (raw pt), same colours.
- `other / singletons` and `other / excluded (non-straight)`: grey context.
- `curved / groups (distinct colour)` (section 3b): **curved chains** -- pieces (straight Vectors, or
  all-`'l'` open polylines whose own turns are <= `MAX_TURN_DEG`) linked by good continuation: each
  link turns by <= `MAX_TURN_DEG` (piece direction and the connector to it), the next piece starts
  within `REACH_FACTOR` x the last piece's length, no gap-consistency test. Greedy from the longest
  seeds; straight chains are included. Plus `curved gradient` / `curved other` like above.
- `curved ratio / ...` (section 3b'): the same chains, plus a dash-length limit -- a piece only joins
  when its length is within `MAX_LEN_RATIO` x the chain's running median member length (so a long
  solid line can't chain into short dashes).
- `corner / ...` (section 3c): >= 2-member collinear groups **merged where their lines meet** --
  both have an endpoint within `CORNER_TOL_PT` of the intersection, or an L-shaped polyline bridges
  the corner (`corner / bridging polylines`); transitive, lines >= `MIN_CORNER_ANGLE_DEG` apart.
  `corner / intersection points` marks every accepted corner.

Gradient ranges are page-wide and only cover groups with >= 2 members.

## 0 - Config

In [ ]:
PDF_PATH = None         # None -> first PDF under references/
PAGE = 0
ANGLE_TOL_DEG = 1.0     # single-linkage angle tolerance (folded to [0, 180))
OFFSET_TOL_PT = 0.5     # max perpendicular-offset span of one collinear group
STRAIGHT_TOL_PT = 0.25  # max point distance from the fitted line for a Vector to count as straight
MAX_TURN_DEG = 15.0     # curved chains: max turn per link and per polyline vertex
REACH_FACTOR = 3.0      # curved chains: next piece within this x the last piece's length
MAX_LEN_RATIO = 2.0     # curved ratio chains: piece length vs chain's running median length
CORNER_TOL_PT = 1.0     # corner merge: endpoint / bridge-vertex distance to the intersection
MIN_CORNER_ANGLE_DEG = 10.0  # corner merge: lines closer to parallel never merge
LINE_WIDTH = 1.0

## 1 - Bootstrap

In [ ]:
import colorsys
import os, sys
from pathlib import Path

_root = os.path.abspath(os.path.join('../..'))
if _root not in sys.path:
    sys.path.insert(0, _root)

import matplotlib.pyplot as plt
import numpy as np

from rastervec.commons.helpers.geometry import union_bbox
from rastervec.notebooks import _vector_probe_helpers as h

if PDF_PATH is None:
    refs = sorted((Path(_root) / 'references').glob('*.pdf'))
    assert refs, 'no PDFs found under references/'
    PDF_PATH = str(refs[0])
print(PDF_PATH, 'page', PAGE)

## 2 - Extract + layer/color/width buckets

In [ ]:
page_meta, vectors = h.load_page_vectors(PDF_PATH, PAGE)
buckets = h.bucket_vectors(vectors)
print(f'{len(vectors)} vectors -> {len(buckets)} layer/color/width buckets')
report = h.DebugReport('dashed_line_collinear_lab', PDF_PATH, page_meta, line_width=LINE_WIDTH)


def collinear_layers(summary, stage):
    """`<stage>` layer: each multi-member group in its own golden-ratio hue."""
    colour = {}
    for g, hue in zip(summary.multi, h.golden_hues(len(summary.multi))):
        rgb = colorsys.hsv_to_rgb(hue, 0.9, 0.85)
        for v, _ in g:
            colour[id(v)] = rgb
    report.add_vectors(stage, 'groups (distinct colour)', '#4a7bd0',
                       [v for g in summary.multi for v, _ in g], lambda v: colour[id(v)])


def top_groups(summary, n=15, turn_of=None):
    rows = sorted(zip(summary.multi, summary.counts, summary.stds), key=lambda r: -r[1])[:n]
    turn_hdr = f" {'turn':>7}" if turn_of else ''
    print(f"{'n':>5} {'angle':>7} {'mean len':>9} {'len std':>8}{turn_hdr}  dashes attr")
    for g, count, std in rows:
        angle = h._mean_axis_angle([fit.angle for _, fit in g])
        mean_len = np.mean([fit.length for _, fit in g])
        turn = f' {turn_of(g):>7.1f}' if turn_of else ''
        print(f'{count:>5} {angle:>7.2f} {mean_len:>9.2f} {std:>8.3f}{turn}  {g[0][0].dashes}')


## 3 - Collinear straight Vectors within each bucket

In [ ]:
v_groups, v_excluded = [], []
bucket_groups, bucket_rest = {}, {}  # per bucket, for the corner merge (section 3c)
for key, members in buckets.items():
    straight, rest = h.split_straight(members, STRAIGHT_TOL_PT)
    v_excluded.extend(rest)
    bucket_rest[key] = rest
    bucket_groups[key] = h.group_collinear(straight, ANGLE_TOL_DEG, OFFSET_TOL_PT)
    v_groups.extend(bucket_groups[key])
v_summary = h.summarise_groups(v_groups)
print(v_summary.describe(len(v_excluded), 'non-straight vector(s)'))

collinear_layers(v_summary, 'collinear')
h.add_gradient_layers(report, v_summary, v_excluded, excluded_label='excluded (non-straight)')
fig = h.group_histograms(v_summary, 'Collinear')
report.save_figure(fig, 'histograms')
plt.show()
top_groups(v_summary)

## 3b - Curved chains (gradual direction drift)

In [ ]:
c_groups, c_excluded = [], []
for key, members in buckets.items():
    pieces, rest = h.split_pieces(members, MAX_TURN_DEG, STRAIGHT_TOL_PT)
    c_excluded.extend(rest)
    c_groups.extend(h.chain_curves(pieces, MAX_TURN_DEG, REACH_FACTOR))
c_summary = h.summarise_groups(c_groups)
print(c_summary.describe(len(c_excluded), 'curve/rect/sharp-polyline vector(s)'))

collinear_layers(c_summary, 'curved')
h.add_gradient_layers(report, c_summary, c_excluded, prefix='curved ',
                      excluded_label='excluded (curve/rect/sharp polyline)')
fig = h.group_histograms(c_summary, 'Curved chain')
report.save_figure(fig, 'curved_histograms')
plt.show()
top_groups(c_summary, turn_of=h.chain_total_turn)


## 3b' - Curved chains + dash-length ratio limit

In [ ]:
r_groups = []
for key, members in buckets.items():
    pieces, _ = h.split_pieces(members, MAX_TURN_DEG, STRAIGHT_TOL_PT)
    r_groups.extend(h.chain_curves(pieces, MAX_TURN_DEG, REACH_FACTOR, max_len_ratio=MAX_LEN_RATIO))
r_summary = h.summarise_groups(r_groups)
print(r_summary.describe(len(c_excluded), 'curve/rect/sharp-polyline vector(s)'))
print(f'vs unconstrained: {len(c_summary.multi)} -> {len(r_summary.multi)} multi-member chains, '
      f'max length std {c_summary.s_hi:.2f} -> {r_summary.s_hi:.2f} pt')

collinear_layers(r_summary, 'curved ratio')
h.add_gradient_layers(report, r_summary, c_excluded, prefix='curved ratio ',
                      excluded_label='excluded (curve/rect/sharp polyline)')
fig = h.group_histograms(r_summary, 'Curved chain (length ratio)')
report.save_figure(fig, 'curved_ratio_histograms')
plt.show()
top_groups(r_summary, turn_of=h.chain_total_turn)


## 3c - Collinear groups merged at intersections

In [ ]:
k_groups, k_counts, k_points, k_bridges = [], [], [], []
for key, groups in bucket_groups.items():
    polylines = [v for v in bucket_rest[key] if h.is_all_lines(v)]
    res = h.merge_at_corners(groups, polylines, angle_tol=ANGLE_TOL_DEG, offset_tol=OFFSET_TOL_PT,
                             corner_tol=CORNER_TOL_PT, min_angle=MIN_CORNER_ANGLE_DEG)
    k_groups.extend(res.groups)
    k_counts.extend(res.source_counts)
    k_points.extend(res.points)
    k_bridges.extend(res.bridges)
print(f'{len(k_groups)} merged component(s) from {sum(k_counts)} collinear groups; '
      f'{len(k_points)} corner(s), {len(k_bridges)} bridging polyline(s)')

colour = {}
for g, hue in zip(k_groups, h.golden_hues(len(k_groups))):
    rgb = colorsys.hsv_to_rgb(hue, 0.9, 0.85)
    for v, _ in g:
        colour[id(v)] = rgb
report.add_vectors('corner', 'merged groups (distinct colour)', '#d07a4a',
                   [v for g in k_groups for v, _ in g], lambda v: colour[id(v)])
report.add_vectors('corner', 'bridging polylines', '#ff00ff', k_bridges, lambda v: (1.0, 0.0, 1.0))
R = 1.5
report.add_boxes('corner', 'intersection points', '#ff0000',
                 [((x - R, y - R, x + R, y + R), (1.0, 0.0, 0.0)) for x, y in k_points])

print(f"{'groups':>6} {'vectors':>8}  bbox")
for g, n in sorted(zip(k_groups, k_counts), key=lambda r: -r[1])[:15]:
    print(f'{n:>6} {len(g):>8}  {tuple(round(c, 1) for c in union_bbox([v.bbox for v, _ in g]))}')


## 4 - Write manifest

In [ ]:
out = report.finish()
print('wrote', out)
for e in report.layers:
    print(f"  {e['stage']:>14} | {e['layer']}")
print()
print('open with:')
print(report.viewer_command())